In [1]:
# standard library imports
from argparse import ArgumentParser
import json
import sys
import logging
logger = logging.getLogger(__name__)

from collections import namedtuple
import ast
import csv
import re

# third party imports
import pandas as pd
from huggingface_hub import login
from unsloth import FastLanguageModel
from tqdm import tqdm

#from json_utils import validate_json, get_schema #extract_json, 

import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:

def clear_gpu_memory():
    import torch, gc, os
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("GPU memory cleared.")

clear_gpu_memory()

GPU memory cleared.


## Main Inference class

The class that retrieves the fine-tuned model from HF

In [3]:
class LLM:
    
    def __init__(self, 
                 model_name:str, 
                 max_seq_length:int,
                 prompt_path:str, 
                 instruct_path:str, 
                 json_fix_path:str, 
                 hf_key_path:str):
        
        self.model, self.tokenizer = self.load_model(model_name,max_seq_length)
        
        self.template = self.load_text(prompt_path)
        self.instruction = self.load_text(instruct_path)
        self.json_fix_instruction = self.load_text(json_fix_path)
        
        self.hf_key = self.load_text(hf_key_path)
        try:
            login(self.hf_key)
        except:
            raise KeyError('Please ensure HuggingFace key is valid')
        
    def load_model(self, model_name:str, max_seq_length:int):
        """Loads the specified modle from Huggingface. Please use a model
        hosted on the Unsloth page.
        
        args:
            model_name (str) : path to model on huggingface e.g. 
            "unsloth/Llama-3.2-3B-Instruct"
        returns:
            FastLanguageModel : unsloth hosted model.
            Tokenizer : corresponding tokenizer for model.
        """
        model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = None,
        load_in_4bit = True)
        model = FastLanguageModel.for_inference(model)
        return model, tokenizer
    
    def load_text(self, path:str):
        """Loads data from a txt file
        
        args:
            path (str) : path to the text data.
        returns:
            str : the text.
        """
        with open(path, 'r') as f:
            text = f.read()
        return text
       
    def get_model_response(self, text:str, location:str, max_tokens=512, max_retries=2, curr_retries=0):
        """Passes the text to the LLM and returns a JSON
        args:
            text (str) : text to be processed.
            location (str) : location relavent to text (e.g. London)
            max_new_tokens (int) : max output size for model.
            max_retries (int) : max number of times to retry if json is broken
            curr_retries (int) : current number of retries performed.  
        returns
            Json formatted list[dict[str, str]]
        """
        prompt = self.template.format(self.instruction, text, "")
        prompt = prompt.replace('', location)
        inputs = self.tokenizer([prompt], return_tensors = "pt").to("cuda")
        response = self.model.generate(**inputs, max_new_tokens = max_tokens)
        output = self.tokenizer.decode(response[0])
        
        ### CHRYSA:  will write the output to file to have a back up and put guardrails to get a resilient response
        
        
        
        # process output and check results
        processed = self.process_output(output)
        # if retries exceeded, reset and return result
        #sys.stdout.write(f'Current Retries = {curr_retries}')
        if curr_retries > max_retries:
            logger.info('Max Retries Exceeded! Attempting LLM fix.')
            processed = self.llm_json_fix(output)
            logger.info(processed)
            curr_retries=0
            return processed
        # if an empty list, return as normal
        elif len(processed)==0:
            curr_retries = 0
            return processed
        # if ['invalid json'] or ['misconstructed json'] retry
        elif isinstance(processed[0], str):
            curr_retries += 1
            max_tokens += 256
            logger.info(f'Current Retries = {curr_retries}')
            logger.info(f'Max Tokens = {max_tokens}')
            logger.info(processed[0])
            
            
            processed = self.get_model_response(text, location, max_tokens, max_retries, curr_retries)
            return processed
        # otherwise we will be ok to return the processed output
        else:
            curr_retries = 0
            return processed
        
    def process_output(self, output:str)->list[dict[str,str]]:
        """Takes a string output from the LLM, extracts the relavent JSON and 
        processes it with reference to the schema defined in 'json_utils'
        
        args: 
            output (str) : a string containing a (possibly misconstructed) json.
        
        returns:
            list[dict[str,str]] : A valid json in accordance with the shema. 
        
        notes:
            Returns ['misconstructed json'] or ['invalid json'] if the json 
            could not be constructed or validated, respectively.
        """
        # get the response part of the output
        response = output.split('')[1:][0]
        # load schema
        schema=get_schema()
        # extract and validate
        try:
            json_out = extract_json(response)
        except:
            return ['misconstructed json']
        try:
            valid_json_out = validate_json(json_out, schema) 
        except:
            return ['invalid json']
        return valid_json_out
    
    def llm_json_fix(self, output):
        """As a last resort, we can ask the LLM to fix a JSON it has produced. 
        """
        json_str = output.split('### Response:')[1:][0]
            
        prompt = self.template.format(self.json_fix_instruction, json_str, '')
        inputs = self.tokenizer([prompt], return_tensors = "pt").to("cuda")
        response = self.model.generate(**inputs, max_new_tokens = 1024)
        output = self.tokenizer.decode(response[0])
        # process output and check results
        processed = self.process_output(output)
        return processed        
        

In [4]:
class LLM_MOD(LLM):

    def get_model_response(self, text:str, location='Edinburgh', max_tokens=128, temprature=0.1, repetition_penalty=1.1,):
        """Passes the text to the LLM and returns a JSON
        args:
            text (str) : text to be processed.
            location (str) : location relavent to text (e.g. London)
            max_new_tokens (int) : max output size for model.
            max_retries (int) : max number of times to retry if json is broken
            curr_retries (int) : current number of retries performed.  
        returns
            Json formatted list[dict[str, str]]
        """
        ## Prepare the prompt that will go to LLM
        prompt = (self.template).format(self.instruction, text, "")
    
        inputs = self.tokenizer(
            prompt,
            return_tensors  = "pt",
            truncation      = True,
            max_length      = 4096
        ).to("cuda")
        
        # Check token count before generating (not necessary for a long run)
        #print(f"Input tokens: {inputs['input_ids'].shape[1]}")

        ## for inference, no need to track opearionts: faster and lower memory usage
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens  = max_tokens,    # max tokens to GENERATE (not total length)
                temperature     = temprature,    # mimimise creativity
                do_sample       = False, 
                repetition_penalty = repetition_penalty,        # higher than 1.1 might be too conservative
            )
        
        # Decode only the new tokens (not the prompt)
        response = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens = True
        )

        final = self.process_output(response) 
        return final

        
    def _extract_json_objects(self,raw: str) -> list[dict]:
        """Extract all valid JSON objects from a messy LLM response."""
        results = []
        # Match anything that looks like a JSON object
        for match in re.finditer(r'\{[^{}]*\}', raw):
            try:
                results.append(json.loads(match.group()))
            except json.JSONDecodeError:
                continue
        return results

    def _deduplicate(self,results: list[dict]) -> list[dict]:
        seen = set()
        unique = []
        for obj in results:
            key = json.dumps(obj, sort_keys=True)
            if key not in seen:
                seen.add(key)
                unique.append(obj)
        return unique

    
    def process_output(self, response:str)->list[dict[str,str]]:
        return self._deduplicate(self._extract_json_objects(response))
    
        
        

In [10]:
## Read the data
def read_listings(in_file,id_col=0,desc_col=1,sep=','):
    """ Parses the CSV fiel with listings and returns
        a dictionary with key the id and value the descripition
        we will pass to the fine-tuned LLM
    """
    Row = namedtuple('Row',['id','desc','scraped','source'])
    out = []
    count_lines,probs = 0,0

    import csv
    with open(in_file, 'r') as fin:
        next(fin)
        reader = csv.reader(fin, delimiter=sep, quoting=csv.QUOTE_MINIMAL)
        for row in reader:
            print(row)
            
            ## clean the source ids and make them unique
            try:
                cleaned_source = set(ast.literal_eval(row[3]))
            except:
                #print(f"**BEWARE: {row[0]}-{count_lines+1} There is '\t' in descr: {row[1]}")
                probs += 1
                cleaned_source = set()

                
            out.append(Row(row[0],row[1],row[2],list(cleaned_source)))            
            count_lines += 1
    
    print(f"{count_lines} rows were processed from listings file {in_file}")
    #print(probs)
    
    #print(f"{len(out)} unique ids were retrieved and will be passed for inference")
    return out
        


## Inference loop

In [6]:
## USER input
model_name = 'axeledi/Qwen3-8B-unsloth-bnb-4bit-abnbTuned-4096'
max_seq_length = 4096
listings_file = "data/LLM_input_without_Teacher_rows_DESCRIPTION_ONLY.csv"


In [11]:
## Step 1: Read the listings
listings = read_listings(listings_file,id_col=0,desc_col=1,sep=",")

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
## Access the description of each record as listings using the name tupple
listings[0]

Row(id='15420', desc='Stunning, spacious ground floor apartment minutes from Princes Street.  Your own ‘home from home’ in an historic property, we strive to offer you the highest standards of comfort and attention to detail.<br /><br />The apartment is completely self-contained with no shared facilities.  It  has its own front door onto the street so there is no shared lobby or stair.<br /><br /><b>The space</b><br />This is a spacious and luxurious apartment which provides comfortable, relaxing accommodation for one or two guests in a great central location.<br /><br />Beautiful 22ft bow-ended (oval) sitting room with curved panelled doors & fireplace, large bedroom quietly located at the rear with 5 ft wide four-poster bed & fireplace, fully equipped kitchen with high quality fittings, bathroom with under-floor heating and bath/shower, free wi-fi (fibre).<br /><br />Located in a central city neighbourhood which has its own identity without feeling touristy, with independent shops, b

### Try the LLM_MOD

In [16]:
model_mod = LLM_MOD(model_name=model_name,
                max_seq_length = max_seq_length,
                prompt_path='prompts/prompt_template.txt',
                instruct_path='prompts/instruction_template.txt',
                json_fix_path='prompts/json_fix_instruction.txt',
                hf_key_path='huggingface_key.txt')

==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.1. vLLM: 0.1.dev1+g18d4e481d.d20260108.cu130.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [43]:
def join_listings_response(listing,response):
    ## Response may have more than one, but these are copies, so we get only the first one
    if len(response) >= 1:
        return listing._asdict() | response[0]
    return listing._asdict() | {'specific_locations':[], 'general_references':[], 'parent_references':[]}


In [51]:
### main loop to collect the results

out = []
n_listings = len(listings)
for i in tqdm(range(1000,1025)):
    resp = model_mod.get_model_response(listings[i].desc)  ## send to LLM and collect the results
    one_dict = join_listings_response(listings[i],resp)    ## merge in one dict
    out.append(one_dict)


    

  0%|          | 0/25 [00:00<?, ?it/s]

Input tokens: 950


  4%|▍         | 1/25 [00:06<02:37,  6.54s/it]

Input tokens: 1105


  8%|▊         | 2/25 [00:13<02:32,  6.63s/it]

...

 92%|█████████▏| 23/25 [01:55<00:10,  5.13s/it]

Input tokens: 1118


 96%|█████████▌| 24/25 [01:58<00:04,  4.48s/it]

Input tokens: 1150


100%|██████████| 25/25 [02:04<00:00,  5.00s/it]


In [36]:
with open('test.json', 'w') as fout:
    json.dump(out , fout)

In [38]:
dat = pd.read_json("test.json")

In [56]:
##### DELETE AFTER THIS FOR DEPLOYMENT

In [19]:
resp = model_mod.get_model_response(listings[11].desc)

Input tokens: 1139


In [18]:
listings[11].desc

'A first for Scotland, The Four Sisters Boatel is a luxury, four star, self catering, canal barge, accommodation located in the very heart of Edinburgh city centre and has been awarded 4 stars by the Scottish Tourist Board.<br /><br /><b>The space</b><br />Fancy something different and a first for Edinburgh? <br /><br />Come and stay with us, we have lowered our prices for the Spring Season!<br /><br />Try The Four Sisters STATIC Boatel; yes a custom built, floating luxury boat right in the very heart of Edinburgh City Centre. Check out our reviews on Trip Advisor!<br /><br />She is tucked away in the Lochrin Basin, measuring a massive 55\' x 10.6" for your sole use. <br /><br />The Four Sisters Boatel is self-catering and can accommodate up to five adults and two children comfortably in two beautifully dressed double cabins (bedrooms) with plasma TV\'s, free view , DVD players and wireless headphones.<br /><br />A large comfortable sofa  in the lounge area with recessed children’s bun

In [30]:
join_listings_response(listings[11],resp)

{'id': '54188',
 'desc': 'A first for Scotland, The Four Sisters Boatel is a luxury, four star, self catering, canal barge, accommodation located in the very heart of Edinburgh city centre and has been awarded 4 stars by the Scottish Tourist Board.<br /><br /><b>The space</b><br />Fancy something different and a first for Edinburgh? <br /><br />Come and stay with us, we have lowered our prices for the Spring Season!<br /><br />Try The Four Sisters STATIC Boatel; yes a custom built, floating luxury boat right in the very heart of Edinburgh City Centre. Check out our reviews on Trip Advisor!<br /><br />She is tucked away in the Lochrin Basin, measuring a massive 55\' x 10.6" for your sole use. <br /><br />The Four Sisters Boatel is self-catering and can accommodate up to five adults and two children comfortably in two beautifully dressed double cabins (bedrooms) with plasma TV\'s, free view , DVD players and wireless headphones.<br /><br />A large comfortable sofa  in the lounge area wit

### For Development

In [15]:
## Download and prepare the model
model = LLM(model_name=model_name,
            max_seq_length = max_seq_length,
            prompt_path='prompts/prompt_template.txt',
            instruct_path='prompts/instruction_template.txt',
            json_fix_path='prompts/json_fix_instruction.txt',
            hf_key_path='huggingface_key.txt')
      

==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.1. vLLM: 0.1.dev1+g18d4e481d.d20260108.cu130.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.1.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [63]:
### Delete it
#print(model.instruction)
#print("-----------------------")
#print(model.template)

In [114]:
def run_inference_desc(model,description,temperature=0.1,repetition_penalty=1.1):
    ## Prepare the prompt that will go to LLM
    prompt = (model.template).format(model.instruction, description, "")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = model.tokenizer(
        prompt,
        return_tensors  = "pt",
        truncation      = True,
        max_length      = 4096
    ).to(device)
    
    # Check token count before generating
    print(f"Input tokens: {inputs['input_ids'].shape[1]}")
    
    with torch.no_grad():
        outputs = model.model.generate(
            **inputs,
            max_new_tokens  = 128,    # max tokens to GENERATE (not total length)
            temperature     = temperature,
            do_sample       = False,
            repetition_penalty = repetition_penalty,
        )
    
    # Decode only the new tokens (not the prompt)
    response = model.tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens = True
    )
    
    return response

In [115]:
response = run_inference_desc(model,listings[679].desc)

Input tokens: 1156


In [93]:
### These functions allow to isolate and exrtact the JSON from the noisy LLM output

import re

def extract_json_objects(raw: str) -> list[dict]:
    """Extract all valid JSON objects from a messy LLM response."""
    results = []
    # Match anything that looks like a JSON object
    for match in re.finditer(r'\{[^{}]*\}', raw):
        try:
            results.append(json.loads(match.group()))
        except json.JSONDecodeError:
            continue
    return results

def deduplicate(results):
    seen = set()
    unique = []
    for obj in results:
        key = json.dumps(obj, sort_keys=True)
        if key not in seen:
            seen.add(key)
            unique.append(obj)
    return unique

In [91]:
for idx in [11,1567,2231,156,12112,13243,7132,178,22657]:
    resp = run_inference_desc(model,listings[idx].desc)
    res = deduplicate(extract_json_objects(resp))
    print(f"{idx} --> {listings[idx].desc}")
    print(resp)
    print(f" => {res}")
    print("-------------------------------------")

Input tokens: 1210
11 --> A first for Scotland, The Four Sisters Boatel is a luxury, four star, self catering, canal barge, accommodation located in the very heart of Edinburgh city centre and has been awarded 4 stars by the Scottish Tourist Board.<br /><br /><b>The space</b><br />Fancy something different and a first for Edinburgh? <br /><br />Come and stay with us, we have lowered our prices for the Spring Season!<br /><br />Try The Four Sisters STATIC Boatel; yes a custom built, floating luxury boat right in the very heart of Edinburgh City Centre. Check out our reviews on Trip Advisor!<br /><br />She is tucked away in the Lochrin Basin, measuring a massive 55' x 10.6" for your sole use. <br /><br />The Four Sisters Boatel is self-catering and can accommodate up to five adults and two children comfortably in two beautifully dressed double cabins (bedrooms) with plasma TV's, free view , DVD players and wireless headphones.<br /><br />A large comfortable sofa  in the lounge area with 

In [129]:
## see the impact of temp and repeat (higher thant 1.1 reduces output)
resp = run_inference_desc(model,listings[22657].desc,temperature=0.1,repetition_penalty=0.9)
res = deduplicate(extract_json_objects(resp))
print(res)

Input tokens: 1050
[{'specific_locations': ['Bruntsfield'], 'general_references': ['city', 'bars', 'cafes', 'restaurants'], 'parent_references': []}]


In [10]:
a = {"instruction":"You will be given a short description from an Airbnb listing in Edinburgh. It may describe either:\n- the property itself, or\n- the surrounding neighbourhood.\n\nYour task is to identify **places or locations described as being near** the property \u2014 for example, anything mentioned as a short walk away, close by, or nearby.\n\nYou should categorise all such places into three groups:\n\n1. `specific_locations` \u2014 locations that are **named and mappable**, such as landmarks, parks, venues, or neighbourhood features (e.g. \"The Meadows\", \"Ocean Terminal\").\n2. `general_locations` \u2014 vague or generic references to places that **are not named or are too broad to geocode**, such as \"train station\", \"shops\", or \"city centre\".\n3. `parent_locations` \u2014 the neighbourhood or area where the property is located (e.g. \"Marchmont\", \"Leith\"). Do not include these in the other two categories unless the location is explicitly described as a separate nearby place. \n\nCorrect minor spelling mistakes for named places (e.g. \"Murayfield\" -> \"Murrayfield Stadium\") so they match real map locations.\n\nFor `general_locations`, strip out descriptive words and articles (e.g. \"the lively bars\" -> \"bars\", \"local shops\" -> \"shops\")..\n\nIf a vague reference clearly refers to a specific place, correct it (e.g. \"the castle\" -> \"Edinburgh Castle\"). But only do this when the reference is **unambiguous**. For example, \"the station\" may refer to multiple locations and should remain general unless disambiguated in the text.\n\nIf a location is mentioned more than once, list it only once in the appropriate category.\n\nReturn your output in the following JSON format, marked \"<|startofjson|>\" and \"<|endofjson|>\" before and after the JSON block:\n\n```\n{\n  \"specific_locations\": [...],\n  \"general_locations\": [...],\n  \"parent_locations\": [...],\n}\n```\n\nHere are some examples of inputs and expected outputs. \n\nExample 1: Vague reference to specific location, reference to parent location.\nDescription:\n\"This bright flat in Marchmont is just a few minutes from the Meadows and near the university.\"\n\nOutput: \n```\n{\n  \"specific_locations\": [\"The Meadows\", \"University of Edinburgh\"],\n  \"general_locations\": [],\n  \"parent_locations\": [\"Marchmont\"]\n}\n```\nExample 2: Vague reference to specific location, reference to parent location, spelling mistake (Lieth -> Leith).\nDescription:\n\"A cosy studio located in Lieth, just steps from The Shore and right beside Ocean Terminal. You're also a short bus ride from the city centre, and less than 30 mins from the airport by tram.\"\n\nOutput:\n```\n{\n  \"specific_locations\": [\"The Shore\", \"Ocean Terminal\", \"Edinburgh International Airport\"],\n  \"general_locations\": [\"city centre\"],\n  \"parent_locations\":[\"Leith\"]\n}\n```\n\nExample 3: Vague references and adjectives.\nDescription:\n\"This cosy apartment is a few minutes\u2019 walk from the station, is handy for local shops and has a great view of the castle.\"\n\nOutput:\n```\n{\n  \"specific_locations\": [\"Edinburgh Castle\"],\n  \"general_locations\": [\"train station\", \"shops\"],\n  \"parent_locations\": []\n}\n```\n\nExample 4: spelling mistake (Princess -> Princes), parent location, ambiguous wording, abbreviation (St -> Street). \nDescription:\n\"Located on Market St, We're just a few minutes' walk from Princess Street, and handy for both the new and old town.\"\n\nOutput:\n```\n{\n  \"specific_locations\": [\"Princes Street\", \"New Town\", \"Old Town\"],\n  \"general_locations\": [],\n  \"parent_locations\": [\"Market Street\"]\n}\n```\n\nExample 5: No locations present.\nDescription:\n\u201cThis conveniently located apartment features high ceilings, modern decor, and a fully equipped kitchen.\u201d\n\nOutput:\n```\n{\n  \"specific_locations\": [],\n  \"general_locations\": [],\n  \"parent_locations\": []\n}\n```\n\nIf you must reason, do so briefly, but always output **only a single JSON object and nothing else**.\n\nNow try with the following property description:\n","input":"Self contained studio 6 minutes by train to Edinburgh's city centre .Located between the city and the vibrant seaside suburb of  Portobello . The Brunstane Studio is the perfect place. <br \/>King sized bed with topper. Free Wi-Fi (7EN74N49) and parking.","response":"[{\"specific_locations\": [\"Portobello\", \"Edinburgh's city centre\"], \"general_references\": [\"train station\"], \"parent_references\": []}]"}

In [11]:
a['input']

"Self contained studio 6 minutes by train to Edinburgh's city centre .Located between the city and the vibrant seaside suburb of  Portobello . The Brunstane Studio is the perfect place. <br \\/>King sized bed with topper. Free Wi-Fi (7EN74N49) and parking."

In [40]:
model.get_model_response(text=a['input'],
                              location='Edinburgh, UK',
                               max_tokens=200, 
                               max_retries=2)

KeyboardInterrupt: 

In [33]:
import os 
instr_path = os.path.expanduser(f"prompts/instruction_template.txt")

with open(instr_path, 'r') as f:
    instruction= f.read()

instruction = instruction.replace('<location>', 'Edinburgh, UK')
print(instruction)

You will be given a short description from an Airbnb listing in Edinburgh. It may describe either:
- the property itself, or
- the surrounding neighbourhood.

Your task is to identify **places or locations described as being near** the property — for example, anything mentioned as a short walk away, close by, or nearby.

You should categorise all such places into three groups:

1. `specific_locations` — locations that are **named and mappable**, such as landmarks, parks, venues, or neighbourhood features (e.g. "The Meadows", "Ocean Terminal").
2. `general_locations` — vague or generic references to places that **are not named or are too broad to geocode**, such as "train station", "shops", or "city centre".
3. `parent_locations` — the neighbourhood or area where the property is located (e.g. "Marchmont", "Leith"). Do not include these in the other two categories unless the location is explicitly described as a separate nearby place. 

Correct minor spelling mistakes for named places (e

In [34]:
ft_prompt = """Below is an instruction that describes a task, paired with an input that provides a specfic example which the task should be applied to. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

In [42]:
val_2 ="The apartment is directly on the Royal Mile about half way between Edinburgh Castle and Holyrood Palace and 400 metres from Waverley Train station. Chrismas Market less than 10 minute walk. Within minutes there are dozens of cafes, bars and restaurants seconds away.  It's a great neighbourhood where you can explore the nearby Arthurs seat or the castle and easily pop back to apartment for a quick rest before going to the vibrant Grassmarket or the Museum of Scotland."

""" ideal response:

"response":"[{\"specific_locations\": [\"Edinburgh Castle\", \"Holyrood Palace\", \"Waverley Train station\", \"Royal Mile\", \"Arthurs Seat\", \"Grassmarket\", \"Museum of Scotland\"], \"general_references\": [\"cafes\", \"bars\", \"Christmas Market\", \"restaurants\"], \"parent_references\": []}]"}

"""

In [44]:
test_4 = "Pleasant double room in 2 bedroom ground floor flat situated near Leith Walk 1.5 km (0.9 mile) North of central shopping street Princes St and served by frequent buses. There is a large dining kitchen and you can cook and use all kitchen facilities.<br /><br /><b>The space</b><br />You will be staying in a pleasant double room in a spacious 2 bedroom ground floor flat which is situated near Leith Walk 1.5 km (just under a mile) North of Edinburgh's Waverley Station. The flat has a large dining kitchen and you are welcome to cook, wash, and use all the kitchen facilities. If you like to take fresh air (or smoke) there is a large shared west facing garden with table, chairs and barbecue facilities provided.  There is fast Wi-Fi throughout the premises. Clean bed-linen and towels are provided, and the bedroom is equipped with cable TV.  We are situated near Leith Walk which has lots of restaurants, and bars, etc.  The flat is serviced by frequent public transport to the centre and other a"

In [45]:
import torch

# Build your prompt exactly as during training
test_input = test_4 #val_2 #a['input']   # your listing description

prompt = ft_prompt.format(instruction, test_input, "")

inputs = model.tokenizer(
    prompt,
    return_tensors  = "pt",
    truncation      = True,
    max_length      = 4096
).to("cuda")

# Check token count before generating
print(f"Input tokens: {inputs['input_ids'].shape[1]}")

with torch.no_grad():
    outputs = model.model.generate(
        **inputs,
        max_new_tokens  = 128,    # max tokens to GENERATE (not total length)
        temperature     = 0.1,
        do_sample       = False,
        repetition_penalty = 1.1,
    )

# Decode only the new tokens (not the prompt)
response = model.tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens = True
)

print(response)

Input tokens: 1142
[{"specific_locations": ["Leith Walk", "Waverley Station", "Princes Street"], "general_references": ["restaurants", "centre", "bars", "public transport"], "parent_references": []}]

